# 6장 — Text-to-SQL 미세 조정

이 노트북은 **Google Colab (T4 GPU)** 환경을 기준으로 작성되었습니다.

주요 변경 사항 (2026 개정):
- `autotrain-advanced` → **`trl.SFTTrainer`**(LoRA 학습) 직접 사용
- GPT-4 평가 → **Gemini 2.0 Flash** (무료 tier, OpenAI 호환 엔드포인트)
- API 키는 Colab **Secrets**(`userdata.get`)에서 안전하게 로드

준비물:
1. **Google AI Studio**에서 무료 API 키 발급 → https://aistudio.google.com/apikey
2. Colab 좌측 🔑 **Secrets** 탭에서 `GEMINI_API_KEY` 이름으로 등록 (Notebook access ON)
3. (선택) `HF_TOKEN`을 같은 방식으로 등록하면 모델 허브 업로드 가능

## 0. 저장소 클론

Colab 런타임에 `utils.py`와 `api_request_parallel_processor.py`를 가져오기 위해 GitHub 저장소를 클론합니다. (이미 클론된 상태라면 자동 스킵됩니다.)

In [ ]:
import os

REPO = 'ch6-sLLM'
if not os.path.isdir(REPO):
    !git clone https://github.com/taejungpark/{REPO}.git
%cd {REPO}
!ls

## 의존성 설치

In [ ]:
!pip install -q \
  transformers==4.49.0 \
  trl==0.13.0 \
  peft==0.14.0 \
  accelerate==1.3.0 \
  bitsandbytes==0.46.0 \
  datasets==3.3.0 \
  tiktoken==0.8.0 \
  aiohttp

## 예제 6.2. SQL 프롬프트

In [ ]:
from utils import make_prompt

print(make_prompt(
    ddl='CREATE TABLE players (player_id INT PRIMARY KEY)',
    question='전체 플레이어 수를 알려주세요.'
))

## 예제 6.4. 평가를 위한 요청 jsonl 작성 함수

`make_requests_for_gpt_evaluation`는 모델 응답을 평가자(LLM judge)에게 채점시키기 위한
JSONL 요청 파일을 만듭니다. 기본 judge는 **`gemini-2.0-flash`**이며, 같은 JSONL은
OpenAI/Ollama/Groq 등 OpenAI 호환 엔드포인트라면 그대로 사용 가능합니다.

In [ ]:
from utils import make_requests_for_gpt_evaluation

## 예제 6.5. Gemini API 키 설정

Colab Secrets에 등록한 `GEMINI_API_KEY`를 `OPENAI_API_KEY` 환경변수로 노출합니다.
(평가 스크립트는 표준 `OPENAI_API_KEY`를 자동으로 인식하므로 별도 인자가 필요 없습니다.)

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('GEMINI_API_KEY')
GEMINI_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/chat/completions'

> **로컬 Ollama로 평가하고 싶다면** (Colab 외부, Mac/Linux):
> ```bash
> ollama pull qwen2.5-coder:7b
> ollama serve  # 백그라운드로 실행
> ```
> 그리고 `GEMINI_URL` 대신 `http://localhost:11434/v1/chat/completions`,
> `make_requests_for_gpt_evaluation(..., model='qwen2.5-coder:7b')`,
> `OPENAI_API_KEY=ollama`로 호출하세요.

## 예제 6.6. 결과 jsonl 파일을 csv로 변환하는 함수

In [ ]:
from utils import change_jsonl_to_csv


## 예제 6.7. 기초 모델로 생성하기

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def make_inference_pipeline(model_id):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    return pipeline('text-generation', model=model, tokenizer=tokenizer)

model_id = 'beomi/Yi-Ko-6B'
hf_pipe = make_inference_pipeline(model_id)

example = """당신은 SQL을 생성하는 SQL 봇입니다. DDL의 테이블을 활용한 Question을 해결할 수 있는 SQL 쿼리를 생성하세요.

### DDL:
CREATE TABLE players (
  player_id INT PRIMARY KEY AUTO_INCREMENT,
  username VARCHAR(255) UNIQUE NOT NULL,
  email VARCHAR(255) UNIQUE NOT NULL,
  password_hash VARCHAR(255) NOT NULL,
  date_joined DATETIME NOT NULL,
  last_login DATETIME
);

### Question:
사용자 이름에 'admin'이 포함되어 있는 계정의 수를 알려주세요.

### SQL:
"""

hf_pipe(example, do_sample=False, return_full_text=False, max_length=512, truncation=True)

## 예제 6.8. 기초 모델 성능 측정

In [ ]:
!mkdir -p results requests


In [ ]:
from datasets import load_dataset

df = load_dataset('shangrilar/ko_text2sql', 'origin')['test'].to_pandas()
for idx, row in df.iterrows():
    df.loc[idx, 'prompt'] = make_prompt(row['context'], row['question'])

gen_sqls = hf_pipe(
    df['prompt'].tolist(),
    do_sample=False, return_full_text=False, max_length=512, truncation=True,
)
df['gen_sql'] = [x[0]['generated_text'] for x in gen_sqls]

eval_filepath = 'text2sql_evaluation.jsonl'
make_requests_for_gpt_evaluation(df, eval_filepath)

Gemini judge를 호출합니다. 무료 tier(15 RPM)에 맞춰 `--max_requests_per_minute 15`로 제한했습니다.
`text2sql_evaluation.jsonl`이 약 100건이라면 7~8분 정도 소요됩니다.

In [ ]:
!python api_request_parallel_processor.py \
    --requests_filepath requests/{eval_filepath} \
    --save_filepath results/{eval_filepath} \
    --request_url {GEMINI_URL} \
    --max_requests_per_minute 15 \
    --max_tokens_per_minute 1000000 \
    --token_encoding_name cl100k_base \
    --max_attempts 5 \
    --logging_level 20

In [ ]:
import json

base_eval = change_jsonl_to_csv(
    f'results/{eval_filepath}', 'results/yi_ko_6b_eval.csv', 'prompt', 'resolve_yn',
)
base_eval['resolve_yn'] = base_eval['resolve_yn'].apply(lambda x: json.loads(x)['resolve_yn'])
num_correct_answers = base_eval.query("resolve_yn == 'yes'").shape[0]
print(f'기초 모델 정답 수: {num_correct_answers} / {len(base_eval)}')

## 예제 6.9. 학습 데이터 불러오기

In [ ]:
from datasets import load_dataset

df_sql = load_dataset('shangrilar/ko_text2sql', 'origin')['train'].to_pandas()
df_sql = df_sql.dropna().sample(frac=1, random_state=42)
df_sql = df_sql.query('db_id != 1')

for idx, row in df_sql.iterrows():
    df_sql.loc[idx, 'text'] = make_prompt(row['context'], row['question'], row['answer'])

!mkdir -p data
df_sql.to_csv('data/train.csv', index=False)

## 예제 6.10. LoRA 미세 조정 (`trl.SFTTrainer`)

`autotrain-advanced` CLI 대신 `trl`의 `SFTTrainer`를 직접 사용합니다.
결과물(LoRA 어댑터)은 동일하며, 의존성/디버깅이 훨씬 깔끔해집니다.

T4(16GB) 기준 메모리 안전을 위해 `per_device_train_batch_size=2`, `gradient_checkpointing=True`로 설정했습니다.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

base_model_name = 'beomi/Yi-Ko-6B'
finetuned_model = 'yi-ko-6b-text2sql'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

train_dataset = load_dataset('csv', data_files='data/train.csv')['train']

sft_config = SFTConfig(
    output_dir=finetuned_model,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy='epoch',
    max_seq_length=1024,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    peft_config=lora_config,
    args=sft_config,
)
trainer.train()
trainer.save_model(finetuned_model)

## 예제 6.11. LoRA 어댑터 결합 및 허깅페이스 허브 업로드

허브 업로드는 선택입니다. `HF_TOKEN`이 Colab Secrets에 등록되어 있어야 동작합니다.

In [ ]:
# 학습 직후 GPU 메모리 비우기 (fp16 재로드 위해 필요)
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={'': 0},
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, finetuned_model).merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# 허브 업로드 (선택)
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN') if 'HF_TOKEN' in userdata.list_keys() else None
if hf_token:
    login(token=hf_token)
    merged.push_to_hub(finetuned_model, use_temp_dir=False)
    tokenizer.push_to_hub(finetuned_model, use_temp_dir=False)
else:
    print('HF_TOKEN이 Secrets에 없어 허브 업로드를 건너뜁니다.')

## 예제 6.12. 미세 조정한 모델로 예시 데이터에 대한 SQL 생성

In [ ]:
del base, merged
gc.collect()
torch.cuda.empty_cache()

# 허브에 업로드한 경우 본인 계정의 모델 ID로, 아니면 로컬 디렉토리 경로로 로드
model_id = finetuned_model  # 또는 'your-username/yi-ko-6b-text2sql'
hf_pipe = make_inference_pipeline(model_id)

hf_pipe(example, do_sample=False, return_full_text=False, max_length=1024, truncation=True)

## 예제 6.13. 미세 조정한 모델 성능 측정

In [ ]:
gen_sqls = hf_pipe(
    df['prompt'].tolist(),
    do_sample=False, return_full_text=False, max_length=1024, truncation=True,
)
df['gen_sql'] = [x[0]['generated_text'] for x in gen_sqls]

ft_eval_filepath = 'text2sql_evaluation_finetuned.jsonl'
make_requests_for_gpt_evaluation(df, ft_eval_filepath)

In [ ]:
!python api_request_parallel_processor.py \
    --requests_filepath requests/{ft_eval_filepath} \
    --save_filepath results/{ft_eval_filepath} \
    --request_url {GEMINI_URL} \
    --max_requests_per_minute 15 \
    --max_tokens_per_minute 1000000 \
    --token_encoding_name cl100k_base \
    --max_attempts 5 \
    --logging_level 20

In [ ]:
ft_eval = change_jsonl_to_csv(
    f'results/{ft_eval_filepath}',
    'results/yi_ko_6b_finetuned_eval.csv',
    'prompt', 'resolve_yn',
)
ft_eval['resolve_yn'] = ft_eval['resolve_yn'].apply(lambda x: json.loads(x)['resolve_yn'])
num_correct_answers = ft_eval.query("resolve_yn == 'yes'").shape[0]
print(f'미세 조정 모델 정답 수: {num_correct_answers} / {len(ft_eval)}')